# Teste da função build_modeling_dataset

Este notebook valida a lógica da função responsável pela construção
do dataset de modelagem antes da utilização dos dados reais.

In [ ]:
import sys
from pathlib import Path

print("Diretório atual:")
print(Path.cwd())

print("\nDiretório pai:")
print(Path.cwd().parent)

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.build_modeling_dataset import build_modeling_dataset

In [ ]:
print(PROJECT_ROOT)

criar dados artificiais de alunos

In [ ]:
import pandas as pd


alunos_teste = pd.DataFrame(
    {
        "ano": [2024, 2024, 2024],
        "id_aluno": ["A1", "A2", "A4"],
        "id_municipio": ["M1", "M1", "M3"],
        "id_escola": ["E1", "E1", "E3"],
        "rede": ["2", "2", "3"],
        "alfabetizado": ["1", "0", "Sim"],
        "peso_aluno": [1.0, 1.2, 0.8],
    }
)

alunos_teste

criar histórico municipal artificial

In [ ]:
municipio_teste = pd.DataFrame(
    {
        "ano": [2023, 2023],
        "id_municipio": ["M1", "M2"],
        "rede": ["2", "3"],
        "taxa_alfabetizacao": [62.5, 55.0],
        "media_portugues": [750.0, 720.0],
    }
)

municipio_teste

executar a função

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.build_modeling_dataset import build_modeling_dataset

print("Função importada com sucesso.")

In [ ]:
dataset_teste = build_modeling_dataset(
    alunos_df=alunos_teste,
    municipio_df=municipio_teste,
)

dataset_teste

conferir manualmente

In [ ]:
dataset_teste[
    [
        "id_aluno",
        "rede",
        "alfabetizado",
        "taxa_alfabetizacao_2023",
        "media_portugues_2023",
        "historico_2023_disponivel",
    ]
]

validar automaticamente

In [ ]:
assert dataset_teste.shape[0] == 3

assert set(dataset_teste["id_aluno"]) == {
    "A1",
    "A2",
    "A4",
}

assert dataset_teste["id_aluno"].is_unique

assert dataset_teste["alfabetizado"].tolist() == [
    1,
    0,
    1,
]

assert (
    dataset_teste.loc[
        dataset_teste["id_aluno"] == "A4",
        "historico_2023_disponivel",
    ].iloc[0]
    == 0
)

assert (
    dataset_teste.loc[
        dataset_teste["id_aluno"] == "A4",
        "taxa_alfabetizacao_2023",
    ].isna().iloc[0]
)

assert (
    dataset_teste.loc[
        dataset_teste["id_aluno"] == "A1",
        "media_portugues_2023",
    ].iloc[0]
    == 750.0
)

print("Todos os testes passaram.")

Teste mínimo com BigQuery

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.preprocessing.bigquery_reader import run_query

teste = run_query(
    query="SELECT 1 AS teste",
    project_id="projeto-fiap-grupo-x",
)

teste

In [ ]:
from src.preprocessing.bigquery_reader import run_query

teste_bigquery = run_query(
    query="SELECT 1 AS teste",
    project_id="projeto-fiap-grupo-x",
)

teste_bigquery

In [ ]:
from src.preprocessing.queries import (
    QUERY_ALUNOS_2024,
    QUERY_MUNICIPIO_2023,
)

In [ ]:
alunos_amostra = run_query(
    query=QUERY_ALUNOS_2024 + "\nLIMIT 10",
    project_id="projeto-fiap-grupo-x",
)

alunos_amostra

In [ ]:
municipio_amostra = run_query(
    query=QUERY_MUNICIPIO_2023 + "\nLIMIT 10",
    project_id="projeto-fiap-grupo-x",
)

municipio_amostra

validar schema e cobertura antes da extração completa

In [ ]:
print("ALUNOS")
print(alunos_amostra.dtypes)

print("\nMUNICÍPIO")
print(municipio_amostra.dtypes)

In [ ]:
query_validacao_municipio = """
SELECT
    ano,
    rede,
    COUNT(*) AS total_registros,

    COUNTIF(taxa_alfabetizacao IS NULL)
        AS taxa_alfabetizacao_nula,

    COUNTIF(media_portugues IS NULL)
        AS media_portugues_nula,

    COUNTIF(proporcao_aluno_nivel_0 IS NULL)
        AS nivel_0_nulo,

    COUNTIF(proporcao_aluno_nivel_1 IS NULL)
        AS nivel_1_nulo,

    COUNTIF(proporcao_aluno_nivel_2 IS NULL)
        AS nivel_2_nulo,

    COUNTIF(proporcao_aluno_nivel_3 IS NULL)
        AS nivel_3_nulo,

    COUNTIF(proporcao_aluno_nivel_4 IS NULL)
        AS nivel_4_nulo,

    COUNTIF(proporcao_aluno_nivel_5 IS NULL)
        AS nivel_5_nulo,

    COUNTIF(proporcao_aluno_nivel_6 IS NULL)
        AS nivel_6_nulo,

    COUNTIF(proporcao_aluno_nivel_7 IS NULL)
        AS nivel_7_nulo,

    COUNTIF(proporcao_aluno_nivel_8 IS NULL)
        AS nivel_8_nulo

FROM `basedosdados.br_inep_avaliacao_alfabetizacao.municipio`

WHERE ano = 2023

GROUP BY
    ano,
    rede

ORDER BY
    rede
"""

validacao_municipio = run_query(
    query=query_validacao_municipio,
    project_id="projeto-fiap-grupo-x",
)

validacao_municipio

Extração completa

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [ ]:
from src.preprocessing.queries import (
    QUERY_ALUNOS_2024,
    QUERY_MUNICIPIO_2023,
)

from src.preprocessing.bigquery_reader import run_query

from src.preprocessing.build_modeling_dataset import (
    build_modeling_dataset,
)

In [ ]:
print(QUERY_ALUNOS_2024)
print(QUERY_MUNICIPIO_2023)

In [ ]:
from src.preprocessing.queries import (
    QUERY_ALUNOS_2024,
    QUERY_MUNICIPIO_2023,
)

alunos_2024 = run_query(
    query=QUERY_ALUNOS_2024,
    project_id="projeto-fiap-grupo-x",
)

municipio_2023 = run_query(
    query=QUERY_MUNICIPIO_2023,
    project_id="projeto-fiap-grupo-x",
)

print("Alunos 2024:", alunos_2024.shape)
print("Município 2023:", municipio_2023.shape)

Construir modeling_dataset_2024

In [ ]:
modeling_dataset_2024 = build_modeling_dataset(
    alunos_df=alunos_2024,
    municipio_df=municipio_2023,
)

print("Dataset construído.")
print("Shape:", modeling_dataset_2024.shape)

In [ ]:
print("=== VALIDAÇÃO DO DATASET DE MODELAGEM ===")

print("\nShape:")
print(modeling_dataset_2024.shape)

print("\nAlunos duplicados:")
print(modeling_dataset_2024["id_aluno"].duplicated().sum())

print("\nDistribuição do target:")
print(
    modeling_dataset_2024["alfabetizado"]
    .value_counts()
    .sort_index()
)

print("\nDistribuição percentual do target:")
print(
    modeling_dataset_2024["alfabetizado"]
    .value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nDisponibilidade do histórico 2023:")
print(
    modeling_dataset_2024[
        "historico_2023_disponivel"
    ].value_counts().sort_index()
)

print("\nValores nulos:")
print(
    modeling_dataset_2024
    .isna()
    .sum()
    .sort_values(ascending=False)
)

validação final antes de salvar

In [ ]:
# Validações finais do dataset de modelagem

assert modeling_dataset_2024.shape[0] == 1_851_852, \
    "Quantidade inesperada de registros."

assert modeling_dataset_2024["id_aluno"].is_unique, \
    "Existem alunos duplicados."

assert modeling_dataset_2024["alfabetizado"].isna().sum() == 0, \
    "Existem targets nulos."

assert set(modeling_dataset_2024["alfabetizado"].unique()) == {0, 1}, \
    "Target contém valores diferentes de 0 e 1."

assert modeling_dataset_2024["id_municipio"].isna().sum() == 0, \
    "Existem municípios nulos."

assert modeling_dataset_2024["rede"].isna().sum() == 0, \
    "Existem redes nulas."

assert modeling_dataset_2024["peso_aluno"].isna().sum() == 0, \
    "Existem pesos amostrais nulos."

assert (
    modeling_dataset_2024["historico_2023_disponivel"].sum()
    == 1_816_270
), "Quantidade inesperada de alunos com histórico."

assert (
    modeling_dataset_2024["taxa_alfabetizacao_2023"].isna()
    ==
    modeling_dataset_2024["media_portugues_2023"].isna()
).all(), "Features históricas apresentam padrões de ausência diferentes."

assert (
    modeling_dataset_2024["historico_2023_disponivel"]
    ==
    modeling_dataset_2024["taxa_alfabetizacao_2023"].notna().astype("int8")
).all(), "Flag de histórico inconsistente."

print("✅ Todas as validações finais passaram.")

In [ ]:
print(modeling_dataset_2024.columns.tolist())

Salvar o dataset de modelagem

In [ ]:
from pathlib import Path

output_dir = PROJECT_ROOT / "data" / "processed"
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "modeling_dataset_2024.parquet"

modeling_dataset_2024.to_parquet(
    output_path,
    index=False,
)

print(f"Dataset salvo em: {output_path}")

In [ ]:
import pandas as pd

dataset_recarregado = pd.read_parquet(output_path)

print("=== VALIDAÇÃO DO PARQUET ===")
print("Shape:", dataset_recarregado.shape)
print("Duplicidades:", dataset_recarregado["id_aluno"].duplicated().sum())
print("Target nulo:", dataset_recarregado["alfabetizado"].isna().sum())
print("Colunas:", dataset_recarregado.columns.tolist())

In [ ]:
tamanho_mb = output_path.stat().st_size / (1024 ** 2)

print(f"Tamanho do arquivo: {tamanho_mb:.2f} MB")

In [ ]:
import pandas as pd

students_current = pd.read_parquet(
    "../data/processed/modeling_dataset_2024.parquet"
)

print("Shape:", students_current.shape)

print("\nColunas:")
print(students_current.columns.tolist())

print("\nTipos:")
print(students_current.dtypes)

print("\nValores únicos de rede:")
print(
    students_current["rede"]
    .value_counts(dropna=False)
)

print("\nDuplicidades de id_aluno:")
print(
    students_current["id_aluno"]
    .duplicated()
    .sum()
)

print("\nDistribuição do target:")
print(
    students_current["alfabetizado"]
    .value_counts(dropna=False)
)

students_current.head()

In [ ]:
students_base = students_current[
    [
        "ano",
        "id_aluno",
        "id_municipio",
        "id_escola",
        "rede",
        "peso_aluno",
        "alfabetizado",
    ]
].copy()

print("Shape:", students_base.shape)

print("\nColunas:")
print(students_base.columns.tolist())

In [ ]:
NETWORK_MAPPING = {
    "2": "Estadual",
    "3": "Municipal",
    "4": "Privada",
}

students_base["rede"] = (
    students_base["rede"]
    .map(NETWORK_MAPPING)
)

print("Distribuição após mapeamento:")
print(
    students_base["rede"]
    .value_counts(dropna=False)
)

print("\nRedes sem mapeamento:")
print(
    students_base["rede"]
    .isna()
    .sum()
)

In [ ]:
from src.preprocessing.gold_reader import read_gold_table
from src.preprocessing.gold_features import (
    build_consolidated_gold_features,
)

# 1. Recarrega as três Golds
indicador_gold = read_gold_table(
    "indicador_alfabetizacao_municipio"
)

desempenho_gold = read_gold_table(
    "desempenho_alunos_municipio"
)

metas_gold = read_gold_table(
    "comparativo_metas_resultados"
)

print("Golds carregadas:")
print("Indicador:", indicador_gold.shape)
print("Desempenho:", desempenho_gold.shape)
print("Metas:", metas_gold.shape)

# 2. Reconstrói a Gold consolidada
gold_features = build_consolidated_gold_features(
    indicador_gold=indicador_gold,
    desempenho_gold=desempenho_gold,
    metas_gold=metas_gold,
)

print("\nGold consolidada:")
print(gold_features.shape)

In [ ]:
print(students_base.shape)

In [ ]:
feature_coverage = pd.Series({
    "gold_municipal": (
        students_gold_audit["_merge"] == "both"
    ).mean(),

    "participacao_2023": (
        students_gold_audit[
            "percentual_participacao_municipio_2023"
        ].notna()
    ).mean(),

    "desempenho_2023": (
        students_gold_audit[
            "pct_alfabetizados_municipio_2023"
        ].notna()
    ).mean(),

    "metas_2024": (
        students_gold_audit[
            "meta_alfabetizacao_municipio_2024"
        ].notna()
    ).mean(),

    "idhm": (
        students_gold_audit[
            "idhm"
        ].notna()
    ).mean(),
}) * 100

print(
    feature_coverage
    .round(2)
    .astype(str)
    + "%"
)

In [ ]:
from src.preprocessing.build_modeling_dataset import (
    build_modeling_dataset_from_gold,
)

modeling_gold = build_modeling_dataset_from_gold(
    alunos_df=students_current,
    gold_features_df=gold_features,
)

print("Shape:", modeling_gold.shape)

print("\nDuplicidades:")
print(modeling_gold["id_aluno"].duplicated().sum())

print("\nRedes:")
print(modeling_gold["rede"].value_counts(dropna=False))

print("\nTarget:")
print(modeling_gold["alfabetizado"].value_counts(dropna=False))

print("\nDisponibilidade Gold:")
print(
    modeling_gold[
        "gold_historico_disponivel"
    ].value_counts(dropna=False)
)

print("\nColunas:")
print(modeling_gold.columns.tolist())